# 不同列名单追加：可运行检查

仅创建临时合成项目，验证去重、按提供列替换及非法编号拒绝。不会操作真实项目或评分文件。


In [1]:
from pathlib import Path
import subprocess
import textwrap

repo = next(candidate for parent in (Path.cwd(), *Path.cwd().parents)
            for candidate in (parent, parent / "easyqc")
            if (candidate / "core/configuration_service.py").is_file())
python = repo / ".venv" / ("Scripts/python.exe" if __import__("os").name == "nt" else "bin/python")
print(subprocess.check_output([str(python), "--version"], text=True).strip())
fixture = textwrap.dedent("""
    from pathlib import Path
    from tempfile import TemporaryDirectory
    import pandas as pd
    from core.configuration_service import ConfigurationService, ConfigurationError
    from core.project_service import ProjectService
    from core.table_service import TableService
    temporary = TemporaryDirectory(prefix="easyqc_ri14_")
    root = Path(temporary.name)
    service = ConfigurationService(ProjectService(root / "projects.json"), TableService())
    service.create_project("SYNTHETIC", root)
    current = pd.DataFrame({"easyqcid": ["ID1", "ID2"], "kept": ["01", "02"], "shared": ["A", "B"]})
    incoming = pd.DataFrame({"easyqcid": ["ID2", "ID3"], "shared": ["C", "D"], "run": ["01", "02"]})
    service.replace_subjects(current, notify=False)
""")
def probe(source):
    result = subprocess.run([str(python), "-c", fixture + "\n" + textwrap.dedent(source)],
                            cwd=repo, capture_output=True, text=True)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    result.check_returncode()


Python 3.10.17


## 1. 去重：旧行不动，新列保留


In [2]:
probe("""
    print("旧表：", current.to_dict("list"))
    print("导入：", incoming.to_dict("list"))
    summary = service.import_subjects(incoming, mode="append", conflict_policy="deduplicate", notify=False)
    result = service.subjects()
    print(result.to_string(index=False))
    assert result.easyqcid.tolist() == ["ID1", "ID2", "ID3"]
    assert result.loc[1, "kept"] == "02" and result.loc[1, "shared"] == "B"
    assert pd.isna(result.loc[1, "run"]) and result.loc[2, "run"] == "02"
    print(summary)
    temporary.cleanup()
""")


INFO - EasyQC日志系统初始化完成
旧表： {'easyqcid': ['ID1', 'ID2'], 'kept': ['01', '02'], 'shared': ['A', 'B']}
导入： {'easyqcid': ['ID2', 'ID3'], 'shared': ['C', 'D'], 'run': ['01', '02']}
easyqcid kept shared  run
     ID1   01      A <NA>
     ID2   02      B <NA>
     ID3 <NA>      D   02
SubjectImportSummary(mode='append', conflict_policy='deduplicate', current_rows=2, incoming_rows=2, result_rows=3, matching_identities=1, new_identities=1, overlapping_columns=('shared',))


## 2. 替换：只更新导入表提供的列


In [3]:
probe("""
    service.import_subjects(incoming, mode="append", conflict_policy="replace", notify=False)
    result = service.subjects()
    print(result.to_string(index=False))
    assert result.loc[1, "kept"] == "02"
    assert result.loc[1, "shared"] == "C" and result.loc[1, "run"] == "01"
    assert pd.isna(result.loc[2, "kept"])
    temporary.cleanup()
""")


INFO - EasyQC日志系统初始化完成
easyqcid kept shared  run
     ID1   01      A <NA>
     ID2   02      C   01
     ID3 <NA>      D   02


## 3. 缺少编号仍然拒绝，旧名单不变


In [4]:
probe("""
    try:
        service.import_subjects(pd.DataFrame({"run": ["03"]}),
                                mode="append", conflict_policy="replace", notify=False)
    except ConfigurationError as error:
        print(type(error).__name__ + ": " + str(error))
    else:
        raise AssertionError("缺少easyqcid不应写入")
    pd.testing.assert_frame_equal(service.subjects(), current)
    print("旧名单未改变")
    temporary.cleanup()
""")


INFO - EasyQC日志系统初始化完成
ConfigurationError: 质控名单缺少 easyqcid
旧名单未改变


## 边界

列名可不同，但每份名单仍需合法且唯一的easyqcid。替换时，缺少整列会保留旧值；已提供列中的空值则会覆盖旧值。示例只验证合成数据；Qt实际应用、失败保留草稿及数值往返另由自动测试覆盖。
